In [35]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests
import os
import json

In [36]:
# Creating tools

# Get conversion ratio from API
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> str:
    """This function fetches the currency factor between a given base currency and a target currency"""

    api_key=os.environ.get("EXCHANGE_RATE_API")
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()

# Multiply conversion ratio
# InjectedToolArg is used to inform LLM that not provide answer from its data
# This means user/developer is going to provide the data at some point 
@tool
def convert(base_currency_value:int, conversion_rate:Annotated[float, InjectedToolArg]) -> float:
    """Converts the value of base currency with conversion rate"""
    return base_currency_value*conversion_rate

In [37]:
get_conversion_factor.invoke({'base_currency':'INR', 'target_currency':'USD'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1776643201,
 'time_last_update_utc': 'Mon, 20 Apr 2026 00:00:01 +0000',
 'time_next_update_unix': 1776729601,
 'time_next_update_utc': 'Tue, 21 Apr 2026 00:00:01 +0000',
 'base_code': 'INR',
 'target_code': 'USD',
 'conversion_rate': 0.01078}

In [38]:
convert.invoke({'base_currency_value':100,'conversion_rate':92.78})

9278.0

In [39]:
# Tool Binding
llm = ChatOpenAI()
model = llm.bind_tools([get_conversion_factor,convert])

In [40]:
history = []

while True:
    message = input("YOU: ")
    #print(HumanMessage(message))
    history.append(HumanMessage(message))
    if message == 'bye':
        break
    message = model.invoke(history)
    #print(message)
    history.append(message)
    # Check if AI called any Tool
    while len(message.tool_calls) > 0:

        # Looping through all the tools
        for tool in message.tool_calls:

            # Execute the tool if the AI gave tool_name as 'get_conversion_factor'
            if tool['name'] == 'get_conversion_factor':
                tool_message = get_conversion_factor.invoke(tool)
                # Adding tool message in history
                #print(tool_message)
                history.append(tool_message)
                # Converting JSON to dict and storing conversion_rate value
                conversion_rate = json.loads(tool_message.content)['conversion_rate']

            # Execute the tool if AI gave tool_name as 'convert'    
            if tool['name'] == 'convert':
                # Fetch the current conversion_rate
                tool['args']['conversion_rate'] = conversion_rate
                tool_message = convert.invoke(tool)
                # Adding tool message in history
                #print(tool_message)
                history.append(tool_message)
                
        # Passing the tool outputs to LLM
        message = model.invoke(history)
        history.append(message)
    print("AI: ",message.content)


AI:  Hello! How can I assist you today?
AI:  100 Indian Rupees is equivalent to 1.078 US Dollars.
AI:  The current conversion rate for 1 Indian Rupee (INR) to US Dollar (USD) is 0.01078.
AI:  Goodbye! Have a great day!
AI:  Goodbye! If you have any more questions in the future, feel free to ask.


In [41]:
history

[HumanMessage(content='Hi ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 90, 'total_tokens': 100, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DWqcTQJhYozLVHMP9Mqa27fHlqxBn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dacd5-08a5-7131-9f34-5f5c2d8d2b52-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 10, 'total_tokens': 100, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 HumanMessage(content='Convert 100